In [6]:
from extracter.pdf_extracter import extract_pdf_to_markdown
path_to_md=extract_pdf_to_markdown("/Users/prithivi/Documents/CODES/zamp_project/data/invoice_1_happy_path.pdf","parsed/")


2026-05-12 20:13:40,657 | INFO | Starting PDF extraction process
2026-05-12 20:13:40,658 | INFO | Input PDF: /Users/prithivi/Documents/CODES/zamp_project/data/invoice_1_happy_path.pdf
2026-05-12 20:13:40,659 | INFO | Output directory: parsed
2026-05-12 20:13:40,659 | INFO | Running opendataloader_pdf.convert()


May 12, 2026 8:13:40 PM org.opendataloader.pdf.processors.DocumentProcessor preprocessing
INFO: File name: /Users/prithivi/Documents/CODES/zamp_project/data/invoice_1_happy_path.pdf
May 12, 2026 8:13:40 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Number of pages: 1
May 12, 2026 8:13:41 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Author: null
May 12, 2026 8:13:41 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Title: 
May 12, 2026 8:13:41 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Creation date: D:20260511112722Z
May 12, 2026 8:13:41 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Modification date: null
May 12, 2026 8:13:41 PM org.opendataloader.pdf.processors.HybridDocumentProcessor processDocument
INFO: Starting hybrid processing for 1 pages
May 12, 2026 8:13:41 PM org.opendataloader.pdf.processors.

2026-05-12 20:13:45,175 | INFO | Extraction completed successfully
2026-05-12 20:13:45,176 | INFO | Markdown generated: parsed/invoice_1_happy_path.md


May 12, 2026 8:13:45 PM org.opendataloader.pdf.markdown.MarkdownGenerator writeToMarkdown
INFO: Created parsed/invoice_1_happy_path.md


In [7]:
path_to_md

'parsed/invoice_1_happy_path.md'

In [5]:
type(path_to_md)

str

In [8]:
from extracter.md_to_json import  extract_invoice_from_markdown_file , structure_scanned_invoice_data

In [9]:
json_op=extract_invoice_from_markdown_file(path_to_md)

2026-05-12 20:14:34,320 | INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [10]:
json_op

{'vendor_name': 'ACME CORPORATION',
 'vendor_tax_id': '94-1234567',
 'vendor_contact': 'Phone: (415) 555-0123 | Email: billing@acmecorp.com',
 'invoice_number': 'INV-8473',
 'invoice_date': '2026-03-15',
 'due_date': '2026-04-14',
 'po_reference': 'PO-2847',
 'bill_to': {'company': 'MidCorp Inc.',
  'address': '456 Tech Avenue, Austin, TX 78701, United States'},
 'line_items': [{'description': 'Cloud Storage Services -',
   'quantity': '500 GB',
   'unit_price': 0.2,
   'amount': 100.0}],
 'subtotal': 100.0,
 'tax': 81.8,
 'tax_rate': 0.18,
 'total_amount': 118.0,
 'currency': '$',
 'payment_terms': 'Net 30 Days',
 'payment_instructions': 'Wire transfer to Account #9876543210, Routing #121000248',
 'extraction_metadata': {'extraction_method': 'ollama_llama3',
  'model': 'llama3:latest',
  'warnings': ['Amount mismatch: subtotal + tax = 181.8, but total = 118.0']}}

In [11]:
from validation.database import InvoiceDatabase
    
db = InvoiceDatabase("data/po_and_payment_database.xlsx")

✅ Loaded 10 Purchase Orders
✅ Loaded 5 Payment History records


In [12]:
from validation.matcher import find_matching_po

matched_po=find_matching_po(json_op, db)

2026-05-12 20:14:42,900 | INFO | Matching invoice: INV-8473
2026-05-12 20:14:42,900 | INFO | Searching exact PO match: PO-2847
2026-05-12 20:14:42,914 | INFO | Exact PO match found: PO-2847


In [13]:
from validation.validator import validate_invoice

validation_result = validate_invoice(
    json_op,
    matched_po,
    db
)

In [14]:
validation_result

{'is_valid': True,
 'issues': [],
 'warnings': [],
 'checks_passed': ['✅ PO match found',
  '✅ PO status: Open',
  '✅ Vendor match',
  '✅ Amount match: $118.00 vs $120.00 (within 5% tolerance)',
  '✅ No duplicate found',
  '✅ All required fields present'],
 'po_details': {'po_number': 'PO-2847',
  'approved_amount': 120,
  'invoice_amount': 118.0,
  'amount_difference': -2.0,
  'amount_difference_percent': 0.016666666666666666}}

In [15]:
from validation.decision import make_decision

final_decision = make_decision(
    json_op,
    matched_po,
    validation_result,
    use_ai_review=True
)

2026-05-12 20:14:58,412 | INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [16]:
final_decision

{'status': 'APPROVED',
 'reasoning': 'Invoice INV-8473 matches PO-2847. Amount $118.00 within approved $120.00. All validations passed.',
 'action': 'Proceed to payment',
 'confidence': 1.0,
 'payment_details': {'amount': 118.0,
  'vendor': 'ACME CORPORATION',
  'po_number': 'PO-2847',
  'payment_terms': 'Net 30 Days',
  'due_date': '2026-04-14'},
 'ai_review': {'summary': 'Invoice INV-8473 matches PO-2847 with an approved amount of $120.00, and the invoice amount of $118.00 falls within the approved amount (5% tolerance). All validations passed.',
  'risk_level': 'LOW',
  'recommended_action': 'Proceed to payment',
  'reviewer_notes': 'Invoice appears valid, no discrepancies found. Please verify payment details before processing.',
  'business_impact': 'Minimal'}}

In [17]:
from RAG.chunker import build_invoice_chunk
result = build_invoice_chunk(final_decision)

2026-05-12 20:27:26,007 | INFO | Starting invoice chunk generation
2026-05-12 20:27:26,008 | INFO | Chunk successfully created for invoice: INV-8473


In [20]:
from pprint import pprint    
pprint(result)

{'chunk': 'Invoice INV-8473 from vendor ACME CORPORATION linked to purchase '
          'order PO-2847 has status APPROVED. Invoice amount is $118.00 with '
          "payment terms 'Net 30 Days'. Due date for payment is 2026-04-14. "
          'Risk level associated with this invoice is LOW. Recommended action '
          "is 'Proceed to payment'. Validation reasoning: Invoice INV-8473 "
          'matches PO-2847. Amount $118.00 within approved $120.00. All '
          'validations passed. AI review summary: Invoice INV-8473 matches '
          'PO-2847 with an approved amount of $120.00, and the invoice amount '
          'of $118.00 falls within the approved amount (5% tolerance). All '
          'validations passed. Reviewer notes: Invoice appears valid, no '
          'discrepancies found. Please verify payment details before '
          'processing. Business impact assessment: Minimal',
 'id': 'INV-8473_PO-2847',
 'metadata': {'action': 'Proceed to payment',
              'amoun

In [21]:
type(result)

dict

In [22]:
from RAG.embedder import InvoiceVectorDB

In [23]:
vector_db = InvoiceVectorDB()

2026-05-12 20:35:28,428 | INFO | Initializing InvoiceVectorDB
2026-05-12 20:35:28,429 | INFO | Vector DB directory ready: vectordatabase
2026-05-12 20:35:28,430 | INFO | Loading embedding model: all-MiniLM-L6-v2
2026-05-12 20:35:28,479 | INFO | No device provided, using mps
2026-05-12 20:35:28,983 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-12 20:35:28,984 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-12 20:35:29,044 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-12 20:35:29,368 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP

In [24]:
result = vector_db.ingest_document(result)

2026-05-12 20:35:59,962 | INFO | Starting document ingestion
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]
2026-05-12 20:36:01,086 | INFO | Vector added to FAISS index
2026-05-12 20:36:01,086 | INFO | Metadata added successfully
2026-05-12 20:36:01,086 | INFO | FAISS index saved successfully
2026-05-12 20:36:01,087 | INFO | Metadata store saved successfully
2026-05-12 20:36:01,087 | INFO | Document ingested successfully: INV-8473_PO-2847


In [26]:
import json 

print("\nIngestion Result:", result)

print("\nDatabase Stats:")
print(json.dumps(vector_db.get_stats(), indent=4))



Ingestion Result: True

Database Stats:
{
    "total_vectors": 1,
    "metadata_records": 1,
    "embedding_dimension": 384,
    "embedding_model": "all-MiniLM-L6-v2",
    "index_path": "vectordatabase/invoice_index.faiss",
    "metadata_path": "vectordatabase/metadata.pkl"
}


In [11]:
from inference.query_understanding import QueryUnderstandingEngine

engine = QueryUnderstandingEngine(
        model="qwen2.5:3b"
    )


query_understanding=engine.understand_query("why was Invoice INV-5567 from TECHVENDOR INC rejected?")

2026-05-12 22:12:50,228 | INFO | QueryUnderstandingEngine initialized with model: qwen2.5:3b
2026-05-12 22:12:50,234 | INFO | Processing user query: why was Invoice INV-5567 from TECHVENDOR INC rejected?
2026-05-12 22:12:50,236 | INFO | Prompt generated successfully
2026-05-12 22:12:53,133 | INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-05-12 22:12:53,135 | INFO | LLM response received
2026-05-12 22:12:53,137 | INFO | Query understanding completed successfully


In [13]:
query_understanding

{'intent': 'invoice_explanation',
 'invoice_number': 'INV-5567',
 'po_number': None,
 'vendor': 'TECHVENDOR INC',
 'status': 'REJECTED',
 'risk_level': None,
 'semantic_query': 'reason for invoice rejection TECHVENDOR INC INV-5567',
 'metadata_filters': {'invoice_number': 'INV-5567',
  'vendor': 'TECHVENDOR INC'}}

In [12]:
import pickle
from inference.metadata_filtering import MetadataFilterEngine

with open("vectordatabase/metadata.pkl", "rb") as f:
    metadata_store = pickle.load(f)

engine = MetadataFilterEngine(
        metadata_store=metadata_store
    )

filtered_docs = engine.filter_documents(
        query_understanding
    )

print("\nFILTERED DOCUMENTS:\n")

for doc in filtered_docs:

        print(doc)

print("\nFILTER SUMMARY:\n")

print(
        engine.get_filter_summary(
            filtered_docs
        )
    )

2026-05-12 22:12:59,543 | INFO | MetadataFilterEngine initialized with 2 records
2026-05-12 22:12:59,543 | INFO | Starting metadata filtering
2026-05-12 22:12:59,544 | INFO | Starting metadata validation
2026-05-12 22:12:59,545 | INFO | Metadata validation completed: {'invoice_number': True, 'vendor': False}
2026-05-12 22:12:59,546 | WARNING | Invalid filters detected: ['vendor']



FILTERED DOCUMENTS:


FILTER SUMMARY:

{'total_documents': 0, 'statuses': {}, 'vendors': []}


In [10]:
candidate_chunks = [
    doc["chunk"]
    for doc in filtered_docs
]

In [14]:
candidate_chunks

["Invoice INV-8473 from vendor ACME CORPORATION linked to purchase order PO-2847 has status APPROVED. Invoice amount is $118.00 with payment terms 'Net 30 Days'. Due date for payment is 2026-04-14. Risk level associated with this invoice is LOW. Recommended action is 'Proceed to payment'. Validation reasoning: Invoice INV-8473 matches PO-2847. Amount $118.00 within approved $120.00. All validations passed. AI review summary: Invoice INV-8473 matches PO-2847 with an approved amount of $120.00, and the invoice amount of $118.00 falls within the approved amount (5% tolerance). All validations passed. Reviewer notes: Invoice appears valid, no discrepancies found. Please verify payment details before processing. Business impact assessment: Minimal"]

In [5]:
filtered_docs

[{'id': 'INV-8473_PO-2847',
  'chunk': "Invoice INV-8473 from vendor ACME CORPORATION linked to purchase order PO-2847 has status APPROVED. Invoice amount is $118.00 with payment terms 'Net 30 Days'. Due date for payment is 2026-04-14. Risk level associated with this invoice is LOW. Recommended action is 'Proceed to payment'. Validation reasoning: Invoice INV-8473 matches PO-2847. Amount $118.00 within approved $120.00. All validations passed. AI review summary: Invoice INV-8473 matches PO-2847 with an approved amount of $120.00, and the invoice amount of $118.00 falls within the approved amount (5% tolerance). All validations passed. Reviewer notes: Invoice appears valid, no discrepancies found. Please verify payment details before processing. Business impact assessment: Minimal",
  'metadata': {'invoice_number': 'INV-8473',
   'vendor': 'ACME CORPORATION',
   'po_number': 'PO-2847',
   'status': 'APPROVED',
   'amount': 118.0,
   'payment_terms': 'Net 30 Days',
   'due_date': '2026-0

In [9]:

from inference.hybrid_search_answer import HybridRetriever


retriever = HybridRetriever()

result = retriever.retrieve_and_answer(

        user_query=(
            "Why was INV-8473 approved?"
        ),

        semantic_query=(
            query_understanding["semantic_query"]
        ),

        candidate_documents=filtered_docs,

        top_k=3,

        llm_model="qwen2.5:3b"
    )

print("\nFINAL ANSWER:\n")

print(result["final_answer"])

2026-05-12 21:45:20,746 | INFO | Loading embedding model: all-MiniLM-L6-v2
2026-05-12 21:45:20,788 | INFO | No device provided, using mps
2026-05-12 21:45:21,164 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-12 21:45:21,183 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-12 21:45:21,446 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-12 21:45:21,463 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-12 21:45:21,466 | INFO | Loading SentenceT

KeyboardInterrupt: 

In [10]:
from inference.inference import InvoiceInferencePipeline
try:

        pipeline = InvoiceInferencePipeline(
            llm_model="qwen2.5:3b"
        )

        result = pipeline.run(

            user_query=(
                "Show me the approved invoice "
                "for ACME CORPORATION with PO-2847"
            ),

            top_k=3
        )

        print("\n" + "=" * 80)

        print("FINAL ANSWER:\n")

        print(result["final_answer"])

        print("\n" + "=" * 80)

        print("QUERY UNDERSTANDING:\n")

        print(result["query_understanding"])

        print("\n" + "=" * 80)

        print("FILTERED DOCUMENTS:\n")

        print(result["filtered_documents_count"])

except Exception as e:

        print(e)

2026-05-12 21:46:15,890 | INFO | Initializing InvoiceInferencePipeline
2026-05-12 21:46:15,890 | INFO | Vector database validation successful
2026-05-12 21:46:15,891 | INFO | Loading metadata store
2026-05-12 21:46:15,891 | INFO | Metadata store loaded successfully | Records: 1
2026-05-12 21:46:15,892 | INFO | QueryUnderstandingEngine initialized with model: qwen2.5:3b
2026-05-12 21:46:15,892 | INFO | Query understanding engine initialized
2026-05-12 21:46:15,892 | INFO | MetadataFilterEngine initialized with 1 records
2026-05-12 21:46:15,892 | INFO | Metadata filtering engine initialized
2026-05-12 21:46:15,892 | INFO | Loading embedding model: all-MiniLM-L6-v2
2026-05-12 21:46:15,895 | INFO | No device provided, using mps
2026-05-12 21:46:16,207 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-12 21:46:16,229 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache


FINAL ANSWER:

The approved invoice for ACME CORPORATION (Invoice Number: INV-8473) related to PO-2847 is as follows:

- **Vendor**: ACME CORPORATION
- **PO Number**: PO-2847
- **Status**: APPROVED
- **Amount**: $118.00

The invoice details have been verified and no issues were found. The payment terms are "Net 30 Days" with a due date of April 14, 2026.

Recommended action is to proceed with the payment according to the specified terms in the invoice (Net 30 Days).

QUERY UNDERSTANDING:

{'intent': 'invoice_search', 'invoice_number': None, 'po_number': 'PO-2847', 'vendor': 'ACME CORPORATION', 'status': 'APPROVED', 'risk_level': None, 'semantic_query': 'approved invoice for ACME CORPORATION with PO-2847', 'metadata_filters': {'vendor': 'ACME CORPORATION', 'status': 'APPROVED'}}

FILTERED DOCUMENTS:

1


In [ ]:

from RAG.chunk_validater import ChunkValidationEngine
final_decision = {
        'status': 'APPROVED',
        'reasoning': (
            'Invoice INV-8473 matches PO-2847. '
            'Amount $118.00 within approved '
            '$120.00. All validations passed.'
        ),
        'action': 'Proceed to payment',
        'confidence': 1.0,
        'payment_details': {
            'amount': 118.0,
            'vendor': 'ACME CORPORATION',
            'po_number': 'PO-2847',
            'payment_terms': 'Net 30 Days',
            'due_date': '2026-04-14'
        },
        'ai_review': {
            'summary': (
                'Invoice INV-8473 matches PO-2847 '
                'with an approved amount of $120.00.'
            ),
            'risk_level': 'LOW',
            'recommended_action': (
                'Proceed to payment'
            ),
            'reviewer_notes': (
                'Invoice appears valid.'
            ),
            'business_impact': 'Minimal'
        }
    }

validator = ChunkValidationEngine()

is_valid, report = validator.validate(
        final_decision
    )

print("\nVALID:", is_valid)

print("\nREPORT:\n")

from pprint import pprint

pprint(report)

2026-05-12 21:33:33,506 | INFO | Starting chunk validation
2026-05-12 21:33:33,507 | INFO | Chunk validation successful



VALID: True

REPORT:

{'errors': [],
 'metadata': {'confidence': 1.0,
              'po_number': 'PO-2847',
              'risk_level': 'LOW',
              'status': 'APPROVED',
              'vendor': 'ACME CORPORATION'},
 'valid': True,
 'warnings': []}


In [14]:
import pickle

with open("vectordatabase/metadata.pkl", "rb") as f:
    metadata_store = pickle.load(f)

print(type(metadata_store))
print(len(metadata_store))

<class 'list'>
2


In [15]:
for i, item in enumerate(metadata_store):
    print("\n====================")
    print("CHUNK:", i)
    print(item)


CHUNK: 0
{'id': 'INV-8473_PO-2847', 'chunk': "Invoice INV-8473 from vendor ACME CORPORATION linked to purchase order PO-2847 has status APPROVED. Invoice amount is $118.00 with payment terms 'Net 30 Days'. Due date for payment is 2026-04-14. Risk level associated with this invoice is LOW. Recommended action is 'Proceed to payment according to the terms specified in the invoice (Net 30 Days) and payment details.'. Validation reasoning: Invoice INV-8473 matches PO-2847. Amount $118.00 within approved $120.00. All validations passed. AI review summary: Invoice INV-8473 for $118.00 has been APPROVED for payment, matching PO-2847 with an approved amount of $120.00. Reviewer notes: Invoice and PO match exactly, with a small difference in approved and invoice amounts (-$2.00 or 1.67%). All validations passed. Business impact assessment: UNKNOWN", 'metadata': {'invoice_number': 'INV-8473', 'vendor': 'ACME CORPORATION', 'po_number': 'PO-2847', 'status': 'APPROVED', 'amount': 118.0, 'payment_te